### **Data**

[Transactions Data Bank ⛔||⛔ Fraud Detection](https://www.kaggle.com/datasets/qusaybtoush1990/transactions-data-bank-fraud-detection?resource=download)

### **Импорт библиотек**

In [4]:
from pathlib import Path
Path.cwd()
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Markdown, HTML
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import mlflow
import mlflow.catboost
from mlflow.models import infer_signature
from sklearn.metrics import f1_score
from sklearn.metrics import fbeta_score
import os
from sklearn.tree import DecisionTreeClassifier


### **Загрузка данных**

In [5]:
file_id = '1xoDH4u_CzwtpMNU3WA9-9VhwR7dZjaI4'  # из .../file/d/<ID>/view
url = f'https://drive.google.com/uc?export=download&id={file_id}'
df = pd.read_csv(url)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048574 entries, 0 to 1048573
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   Date            1048574 non-null  object 
 1   nameOrig        1048574 non-null  object 
 2   amount          1048574 non-null  float64
 3   oldbalanceOrg   1048574 non-null  float64
 4   newbalanceOrig  1048574 non-null  float64
 5   City            1048574 non-null  object 
 6   type            1048574 non-null  object 
 7   Card Type       1048574 non-null  object 
 8   Exp Type        1048574 non-null  object 
 9   Gender          1048574 non-null  object 
 10  isFraud         1048574 non-null  int64  
dtypes: float64(3), int64(1), object(7)
memory usage: 88.0+ MB


**Пока из модели удалим "Date" и "nameOrig":**

In [7]:
df_new = df.copy()
df_new = df_new.drop(['Date','nameOrig'],axis=1)
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048574 entries, 0 to 1048573
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   amount          1048574 non-null  float64
 1   oldbalanceOrg   1048574 non-null  float64
 2   newbalanceOrig  1048574 non-null  float64
 3   City            1048574 non-null  object 
 4   type            1048574 non-null  object 
 5   Card Type       1048574 non-null  object 
 6   Exp Type        1048574 non-null  object 
 7   Gender          1048574 non-null  object 
 8   isFraud         1048574 non-null  int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 72.0+ MB


**Сделаем one-hot encoding для всех признаков, кроме "City"**

In [8]:
df_new_encoded = pd.get_dummies(df_new, columns=['type','Card Type', 'Exp Type', 'Gender'], drop_first=True)
df_new_encoded.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048574 entries, 0 to 1048573
Data columns (total 23 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   amount                   1048574 non-null  float64
 1   oldbalanceOrg            1048574 non-null  float64
 2   newbalanceOrig           1048574 non-null  float64
 3   City                     1048574 non-null  object 
 4   isFraud                  1048574 non-null  int64  
 5   type_CASH_OUT            1048574 non-null  bool   
 6   type_DEBIT               1048574 non-null  bool   
 7   type_PAYMENT             1048574 non-null  bool   
 8   type_TRANSFER            1048574 non-null  bool   
 9   Card Type_Gold           1048574 non-null  bool   
 10  Card Type_Mass           1048574 non-null  bool   
 11  Card Type_Platinum       1048574 non-null  bool   
 12  Card Type_Signature      1048574 non-null  bool   
 13  Card Type_Silver         1048574 non-null 

**Разделим датасет на трейн и тест 70\30**

In [9]:
X = df_new_encoded.drop('isFraud', axis=1)
y = df_new_encoded['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y)

**Стандартизируем вещественные признаки:**

In [10]:
scaler = StandardScaler()

X_train[['amount','oldbalanceOrg','newbalanceOrig']] = scaler.fit_transform(X_train[['amount','oldbalanceOrg','newbalanceOrig']])
X_test[['amount','oldbalanceOrg','newbalanceOrig']] = scaler.transform(X_test[['amount','oldbalanceOrg','newbalanceOrig']])

**Далее 'City' перекодируем с помощью target encoding**

In [11]:
FEATURE = 'City'
TARGET = 'isFraud'
MIN_SAMPLES_LEAF = 50
SMOOTHING = 20

global_mean = y_train.mean()
print(f"Глобальное среднее мошенничества (для сглаживания): {global_mean:.4f}")

temp_df = X_train[[FEATURE]].copy()
temp_df[TARGET] = y_train

agg = temp_df.groupby(FEATURE)[TARGET].agg(['count', 'mean'])
agg.columns = ['counts', 'mean_target']

# применение сглаживания
agg['lambda'] = 1 / (1 + np.exp((MIN_SAMPLES_LEAF - agg['counts']) / SMOOTHING))

agg['smoothed_target'] = agg['lambda'] * agg['mean_target'] + (1 - agg['lambda']) * global_mean

city_encoding_map = agg['smoothed_target'].to_dict()

X_train[f'{FEATURE}_TargetEncoded'] = X_train[FEATURE].map(city_encoding_map).fillna(global_mean)
X_test[f'{FEATURE}_TargetEncoded'] = X_test[FEATURE].map(city_encoding_map)


X_test[f'{FEATURE}_TargetEncoded'] = X_test[f'{FEATURE}_TargetEncoded'].fillna(global_mean) #  обработка непредставленных категорий в тесте


X_train = X_train.drop(FEATURE, axis=1)
X_test = X_test.drop(FEATURE, axis=1)

print("\nTarget Encoding завершен.")
print(f"Первые 5 значений нового признака в X_train: \n{X_train[f'{FEATURE}_TargetEncoded'].head()}")

Глобальное среднее мошенничества (для сглаживания): 0.1676

Target Encoding завершен.
Первые 5 значений нового признака в X_train: 
230565    0.171145
982532    0.169802
723425    0.169234
660727    0.170948
736648    0.171145
Name: City_TargetEncoded, dtype: float64


**Финальные фичи в трейне и тесте:**

In [12]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 734001 entries, 230565 to 293268
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   amount                   734001 non-null  float64
 1   oldbalanceOrg            734001 non-null  float64
 2   newbalanceOrig           734001 non-null  float64
 3   type_CASH_OUT            734001 non-null  bool   
 4   type_DEBIT               734001 non-null  bool   
 5   type_PAYMENT             734001 non-null  bool   
 6   type_TRANSFER            734001 non-null  bool   
 7   Card Type_Gold           734001 non-null  bool   
 8   Card Type_Mass           734001 non-null  bool   
 9   Card Type_Platinum       734001 non-null  bool   
 10  Card Type_Signature      734001 non-null  bool   
 11  Card Type_Silver         734001 non-null  bool   
 12  Exp Type_Entertainment   734001 non-null  bool   
 13  Exp Type_Food            734001 non-null  bool   
 14  Exp 

---


#### **Обучение**

In [14]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Если на сервере включена basic-auth, задайте логин/пароль
# os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "admin")
# os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "password")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("MLflow URI:", mlflow.get_tracking_uri())

MLflow URI: http://localhost:5050


In [15]:
# Быстрая проверка подключения
mlflow.search_experiments(max_results=3)

[<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1779728791796, experiment_id='1', last_update_time=1779728791796, lifecycle_stage='active', name='fraud-detection-baseline-linear-regression', tags={}>,
 <Experiment: artifact_location='s3://mlflow-bucket/mlflow/0', creation_time=1779726783799, experiment_id='0', last_update_time=1779726783799, lifecycle_stage='active', name='Default', tags={}>]

In [16]:
from mlflow.tracking import MlflowClient
PROJECT_NAME = "fraud-detection"
experiment_name = f"{PROJECT_NAME}"
artifact_location = "mlflow-artifacts:/"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = f"{PROJECT_NAME}-{exp_id}"

Created experiment: 2


##### **Бейзлайн: логистичческая регрессия**

In [17]:
params = {
        "solver": "saga",
        "max_iter": 2000,
        "random_state": 42
    }

with mlflow.start_run(experiment_id=exp_id, run_name="Baseline. Logistic Regression"):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())

    log_reg_model = LogisticRegression(**params)
    log_reg_model.fit(X_train, y_train)

    y_pred = log_reg_model.predict(X_test)
    proba = log_reg_model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall": recall_score(y_test, y_pred, pos_label=1),
        "f1_score": f1_score(y_test, y_pred, pos_label=1),
        "f2_score": fbeta_score(y_test, y_pred, pos_label=1, beta=2),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
    }

    conf_matrix = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("Предсказано")
    ax.set_ylabel("Истинно")
    ax.set_title("Матрица ошибок")
    plt.tight_layout()

    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test, log_reg_model.predict_proba(X_test))

    model_info = mlflow.sklearn.log_model(
        sk_model=log_reg_model,
        artifact_path="model",
        signature=signature,
        registered_model_name=registered_model_name,
    )

    mlflow.log_input(
        mlflow.data.from_pandas(df, source="../data/raw/Fraud.csv"),
        context="training",
    )

    client = MlflowClient()
    new_version = model_info.registered_model_version

    client.set_model_version_tag(
        registered_model_name,
        new_version,
        "env",
        "training",
    )

    client.set_registered_model_alias(
        registered_model_name,
        "training",
        new_version,
    )

    run_id = mlflow.active_run().info.run_id

    print("Run ID:", run_id)
    print("Registered model version:", new_version)
    print("Alias 'training' points to version:", new_version)
    print("Metrics:", metrics)

Artifact URI for this run: mlflow-artifacts:/de6c7821ab7c42a7a30e45545085e401/artifacts


2026/05/25 21:19:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'fraud-detection-2'.
2026/05/25 21:19:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-2, version 1
Created version '1' of model 'fraud-detection-2'.
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '../data/raw/Fraud.csv'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactData

Run ID: de6c7821ab7c42a7a30e45545085e401
Registered model version: 1
Alias 'training' points to version: 1
Metrics: {'accuracy': 0.8622259380175666, 'precision': 0.6841630733045865, 'recall': 0.3309503944174757, 'f1_score': 0.4461058712266442, 'f2_score': 0.369056983599347, 'roc_auc': 0.8325546001211345, 'pr_auc': 0.486031654344674}
🏃 View run Baseline. Logistic Regression at: http://localhost:5050/#/experiments/2/runs/de6c7821ab7c42a7a30e45545085e401
🧪 View experiment at: http://localhost:5050/#/experiments/2


In [18]:
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nМатрица Ошибок")
print(conf_matrix)

print("\nОтчет по классификации")
print(classification_report(y_test, y_pred))


Матрица Ошибок
[[253780   8057]
 [ 35283  17453]]

Отчет по классификации
              precision    recall  f1-score   support

           0       0.88      0.97      0.92    261837
           1       0.68      0.33      0.45     52736

    accuracy                           0.86    314573
   macro avg       0.78      0.65      0.68    314573
weighted avg       0.85      0.86      0.84    314573



##### ***Модель №1. Логистическая регрессия с балансировокй классов***

Полнота (recall): Модель смогла обнаружить 33% действительно фродовых операций - Это очень мало

Прецизионность (precision): Из всех транзакций, которые модель пометила как фрод, 68% действительно им являются. Т.е. модель довольно много нефродовых операций считает фродовыми


Теперь будем балансировать классы:

In [19]:
params = {
    "class_weight": "balanced",
    "penalty": "l2",
    "solver": "saga",
    "max_iter": 2000,
    "random_state": 42
}

with mlflow.start_run(experiment_id=exp_id, run_name="Model #1. Logistic Regression with Balanced Class Weights"):
        print("Artifact URI for this run:", mlflow.get_artifact_uri())

        log_reg_balanced = LogisticRegression(**params)
        log_reg_balanced.fit(X_train, y_train)
        y_pred_balanced = log_reg_balanced.predict(X_test)
        y_pred = log_reg_balanced.predict(X_test)
        proba = log_reg_balanced.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred_balanced),
            "precision": precision_score(y_test, y_pred_balanced, pos_label=1),
            "recall": recall_score(y_test, y_pred_balanced, pos_label=1),
            "f1_score": f1_score(y_test, y_pred_balanced, pos_label=1),
            "f2_score": fbeta_score(y_test, y_pred_balanced, pos_label=1, beta=2),
            "roc_auc": roc_auc_score(y_test, proba),
            "pr_auc": average_precision_score(y_test, proba),
        }

        conf_matrix = confusion_matrix(y_test, y_pred_balanced)

        fig, ax = plt.subplots(figsize=(6, 4))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_xlabel("Предсказано")
        ax.set_ylabel("Истинно")
        ax.set_title("Матрица ошибок")
        plt.tight_layout()

        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

        report = classification_report(y_test, y_pred)

        mlflow.log_text(report, "classification_report.txt")

        signature = infer_signature(X_test, log_reg_balanced.predict_proba(X_test))

        model_info = mlflow.sklearn.log_model(
            sk_model=log_reg_balanced,
            artifact_path="model",
            signature=signature,
            registered_model_name=registered_model_name,
        )

        mlflow.log_input(
            mlflow.data.from_pandas(df, source="../data/raw/Fraud.csv"),
            context="training",
        )

        client = MlflowClient()
        new_version = model_info.registered_model_version

        client.set_model_version_tag(
            registered_model_name,
            new_version,
            "env",
            "training",
        )

        client.set_registered_model_alias(
            registered_model_name,
            "training",
            new_version,
        )

        run_id = mlflow.active_run().info.run_id

        print("Run ID:", run_id)
        print("Registered model version:", new_version)
        print("Alias 'training' points to version:", new_version)
        print("Metrics:", metrics)

Artifact URI for this run: mlflow-artifacts:/0439e82dd7d949b4b87182cf351946d6/artifacts


c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
2026/05/25 21:25:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'fraud-detection-2' already exists. Creating a new version of this model...
2026/05/25 21:25:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-2, version 2
Created version '2' of model 'fraud-detection-2'.
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-pac

Run ID: 0439e82dd7d949b4b87182cf351946d6
Registered model version: 2
Alias 'training' points to version: 2
Metrics: {'accuracy': 0.6526847504394846, 'precision': 0.31738523573200994, 'recall': 0.9313561893203883, 'f1_score': 0.47343460826650213, 'f2_score': 0.6715413895694785, 'roc_auc': 0.8323278964717511, 'pr_auc': 0.4908710187952}
🏃 View run Model #1. Logistic Regression with Balanced Class Weights at: http://localhost:5050/#/experiments/2/runs/0439e82dd7d949b4b87182cf351946d6
🧪 View experiment at: http://localhost:5050/#/experiments/2


In [20]:
conf_matrix = confusion_matrix(y_test, y_pred_balanced)
print("\nМатрица Ошибок")
print(conf_matrix)

print("\nОтчет по Классификации")
print(classification_report(y_test, y_pred_balanced))


Матрица Ошибок
[[156201 105636]
 [  3620  49116]]

Отчет по Классификации
              precision    recall  f1-score   support

           0       0.98      0.60      0.74    261837
           1       0.32      0.93      0.47     52736

    accuracy                           0.65    314573
   macro avg       0.65      0.76      0.61    314573
weighted avg       0.87      0.65      0.70    314573



##### ***Модель №2. Решающее дерево (без баласнировки классов)***

Полнота (recall): Модель смогла обнаружить 93% действительно фродовых операций - Это хорошо

Прецизионность (precision): Из всех транзакций, которые модель пометила как фрод, только 32% действительно им являются. Т.е. мы очень много нефродовых операций считаем фродом

In [21]:
coefficients = log_reg_balanced.coef_[0]
feature_names = X_train.columns
feature_importance = pd.DataFrame({
    'Признак': feature_names,
    'Коэффициент': coefficients})

print("Все Признаки")
print(feature_importance.to_string(index=False))

Все Признаки
                Признак  Коэффициент
                 amount    -0.058683
          oldbalanceOrg    -0.933963
         newbalanceOrig    -0.942472
          type_CASH_OUT     0.439283
             type_DEBIT    -1.560971
           type_PAYMENT    -2.694158
          type_TRANSFER     2.333943
         Card Type_Gold    -0.104330
         Card Type_Mass    -0.507270
     Card Type_Platinum    -0.114440
    Card Type_Signature    -0.103667
       Card Type_Silver    -0.083901
 Exp Type_Entertainment    -0.007681
          Exp Type_Food     0.017201
          Exp Type_Fuel     0.025948
       Exp Type_Grocery    -0.006847
Exp Type_Health_Fitness    -0.099647
          Exp Type_Home    -0.635822
 Exp Type_Personal_Care     0.106375
        Exp Type_Travel    -0.067597
               Gender_M    -0.005426
     City_TargetEncoded     7.523539


Город с исторически высоким уровнем мошенничества значительно повышает шансы на то что операция - фрод

TRANSFER - тоже сильный маркер, что операция фрод

***Теперь постороим дерево CART*** (без балансировки классов)

In [22]:
params = {
    "criterion": "gini",               # можно 'entropy', но Gini — стандарт CART
    "max_depth": 5,                    # ограничиваем глубину
    "min_samples_split": 20,           # минимум 20 объектов для разбиения
    "min_samples_leaf": 10,            # минимум 10 в листе
    "random_state": 42
}

with mlflow.start_run(experiment_id=exp_id, run_name="Model #2. Decision Tree Classifier without balanced class weights"):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())

    clf = DecisionTreeClassifier(**params)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall": recall_score(y_test, y_pred, pos_label=1),
        "f1_score": f1_score(y_test, y_pred, pos_label=1),
        "f2_score": fbeta_score(y_test, y_pred, pos_label=1, beta=2),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
    }

    conf_matrix = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("Предсказано")
    ax.set_ylabel("Истинно")
    ax.set_title("Матрица ошибок")
    plt.tight_layout()

    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    report = classification_report(y_test, y_pred)

    mlflow.log_text(report, "classification_report.txt")

    print("\nМатрица Ошибок")
    print(conf_matrix)

    print("\nОтчет по Классификации")
    print(report)

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test, clf.predict_proba(X_test))

    model_info = mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="model",
        signature=signature,
        registered_model_name=registered_model_name,
    )

    mlflow.log_input(
        mlflow.data.from_pandas(df, source="../data/raw/Fraud.csv"),
        context="training",
    )

    new_version = model_info.registered_model_version

    client.set_model_version_tag(
        registered_model_name,
        new_version,
        "env",
        "training",
    )

    client.set_registered_model_alias(
        registered_model_name,
        "training",
        new_version,
    )

    run_id = mlflow.active_run().info.run_id

    print("Run ID:", run_id)
    print("Registered model version:", new_version)
    print("Alias 'training' points to version:", new_version)
    print("Metrics:", metrics)

Artifact URI for this run: mlflow-artifacts:/ce5a473eb2fd41399b9182f24308a1e8/artifacts

Матрица Ошибок
[[255628   6209]
 [ 35265  17471]]

Отчет по Классификации
              precision    recall  f1-score   support

           0       0.88      0.98      0.92    261837
           1       0.74      0.33      0.46     52736

    accuracy                           0.87    314573
   macro avg       0.81      0.65      0.69    314573
weighted avg       0.86      0.87      0.85    314573



2026/05/25 21:25:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'fraud-detection-2' already exists. Creating a new version of this model...
2026/05/25 21:25:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-2, version 3
Created version '3' of model 'fraud-detection-2'.
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '../data/raw/Fraud.csv'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: Loc

Run ID: ce5a473eb2fd41399b9182f24308a1e8
Registered model version: 3
Alias 'training' points to version: 3
Metrics: {'accuracy': 0.8681577884942446, 'precision': 0.7377956081081081, 'recall': 0.3312917172330097, 'f1_score': 0.4572602596314908, 'f2_score': 0.37231911483906166, 'roc_auc': 0.8795265441476487, 'pr_auc': 0.5463862062059461}
🏃 View run Model #2. Decision Tree Classifier without balanced class weights at: http://localhost:5050/#/experiments/2/runs/ce5a473eb2fd41399b9182f24308a1e8
🧪 View experiment at: http://localhost:5050/#/experiments/2


##### ***Модель №3. Решающее дерево (с балансировкой классов)***

Все еще плохо - модель старается повысить точность за счет бОльшего по объему класса

***Теперь постороим дерево CART*** (с балансировкой классов)

In [23]:
params = {
    "criterion": "gini",               # можно 'entropy', но Gini — стандарт CART
    "max_depth": 15,                    # ограничиваем глубину
    "min_samples_split": 20,           # минимум 20 объектов для разбиения
    "min_samples_leaf": 10,            # минимум 10 в листе
    "class_weight": "balanced",
    "random_state": 42
}

with mlflow.start_run(experiment_id=exp_id, run_name="Model #3. Decision Tree Classifier with balanced class weights"):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())

    clf = DecisionTreeClassifier(**params)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall": recall_score(y_test, y_pred, pos_label=1),
        "f1_score": f1_score(y_test, y_pred, pos_label=1),
        "f2_score": fbeta_score(y_test, y_pred, pos_label=1, beta=2),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
    }

    conf_matrix = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("Предсказано")
    ax.set_ylabel("Истинно")
    ax.set_title("Матрица ошибок")
    plt.tight_layout()

    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    report = classification_report(y_test, y_pred)

    mlflow.log_text(report, "classification_report.txt")

    print("\nМатрица Ошибок")
    print(conf_matrix)

    print("\nОтчет по Классификации")
    print(report)

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test, clf.predict_proba(X_test))

    model_info = mlflow.sklearn.log_model(
        sk_model=clf,
        artifact_path="model",
        signature=signature,
        registered_model_name=registered_model_name,
    )

    mlflow.log_input(
        mlflow.data.from_pandas(df, source="../data/raw/Fraud.csv"),
        context="training",
    )

    new_version = model_info.registered_model_version

    client.set_model_version_tag(
        registered_model_name,
        new_version,
        "env",
        "training",
    )

    client.set_registered_model_alias(
        registered_model_name,
        "training",
        new_version,
    )

    run_id = mlflow.active_run().info.run_id

    print("Run ID:", run_id)
    print("Registered model version:", new_version)
    print("Alias 'training' points to version:", new_version)
    print("Metrics:", metrics)

Artifact URI for this run: mlflow-artifacts:/03b3c1af6ee043cca7f93c4702903349/artifacts

Матрица Ошибок
[[186192  75645]
 [  4313  48423]]

Отчет по Классификации
              precision    recall  f1-score   support

           0       0.98      0.71      0.82    261837
           1       0.39      0.92      0.55     52736

    accuracy                           0.75    314573
   macro avg       0.68      0.81      0.69    314573
weighted avg       0.88      0.75      0.78    314573



2026/05/25 21:25:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'fraud-detection-2' already exists. Creating a new version of this model...
2026/05/25 21:25:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-2, version 4
Created version '4' of model 'fraud-detection-2'.
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '../data/raw/Fraud.csv'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: Loc

Run ID: 03b3c1af6ee043cca7f93c4702903349
Registered model version: 4
Alias 'training' points to version: 4
Metrics: {'accuracy': 0.7458205249655883, 'precision': 0.3902940323048651, 'recall': 0.9182152609223301, 'f1_score': 0.5477591004728399, 'f2_score': 0.7227054553269734, 'roc_auc': 0.8785013789515154, 'pr_auc': 0.5687560832915466}
🏃 View run Model #3. Decision Tree Classifier with balanced class weights at: http://localhost:5050/#/experiments/2/runs/03b3c1af6ee043cca7f93c4702903349
🧪 View experiment at: http://localhost:5050/#/experiments/2


***Модель №4***

Балансировка классов и увеличение глубины дерева сдвинули фокус модели с 0-го класса, тем самым предсказывая большинство фродовых операций, но снова в ущерб ложным срабатываниям на легальных транзакций

##### **Модель №4. Случайный лес**

In [25]:
params = {
    "n_estimators": 100,
    "max_depth": 20,
    "random_state": 42
}

with mlflow.start_run(experiment_id=exp_id, run_name="Model #4. Random Forest"):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())

    model_rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
    model_rf.fit(X_train, y_train)

    y_pred = model_rf.predict(X_test)
    proba = model_rf.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall": recall_score(y_test, y_pred, pos_label=1),
        "f1_score": f1_score(y_test, y_pred, pos_label=1),
        "f2_score": fbeta_score(y_test, y_pred, pos_label=1, beta=2),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
    }

    conf_matrix = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("Предсказано")
    ax.set_ylabel("Истинно")
    ax.set_title("Матрица ошибок")
    plt.tight_layout()

    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    report = classification_report(y_test, y_pred)

    mlflow.log_text(report, "classification_report.txt")

    print("\nМатрица Ошибок")
    print(conf_matrix)

    print("\nОтчет по Классификации")
    print(report)

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test, model_rf.predict_proba(X_test))

    model_info = mlflow.sklearn.log_model(
        sk_model=model_rf,
        artifact_path="model",
        signature=signature,
        registered_model_name=registered_model_name,
    )

    mlflow.log_input(
        mlflow.data.from_pandas(df, source="../data/raw/Fraud.csv"),
        context="training",
    )

    new_version = model_info.registered_model_version

    client.set_model_version_tag(
        registered_model_name,
        new_version,
        "env",
        "training",
    )

    client.set_registered_model_alias(
        registered_model_name,
        "training",
        new_version,
    )

    run_id = mlflow.active_run().info.run_id

    print("Run ID:", run_id)
    print("Registered model version:", new_version)
    print("Alias 'training' points to version:", new_version)
    print("Metrics:", metrics)

Artifact URI for this run: mlflow-artifacts:/bc31007ca50c457dba1ea5b6fa1ea6d9/artifacts

Матрица Ошибок
[[255779   6058]
 [ 35566  17170]]

Отчет по Классификации
              precision    recall  f1-score   support

           0       0.88      0.98      0.92    261837
           1       0.74      0.33      0.45     52736

    accuracy                           0.87    314573
   macro avg       0.81      0.65      0.69    314573
weighted avg       0.85      0.87      0.85    314573



2026/05/25 22:19:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'fraud-detection-2' already exists. Creating a new version of this model...
2026/05/25 22:19:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-2, version 6
Created version '6' of model 'fraud-detection-2'.
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '../data/raw/Fraud.csv'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: Loc

Run ID: bc31007ca50c457dba1ea5b6fa1ea6d9
Registered model version: 6
Alias 'training' points to version: 6
Metrics: {'accuracy': 0.8676809516392061, 'precision': 0.7391940761150336, 'recall': 0.3255840412621359, 'f1_score': 0.4520562371649729, 'f2_score': 0.36661086722579983, 'roc_auc': 0.88191873547632, 'pr_auc': 0.5841130379675787}
🏃 View run Model #4. Random Forest at: http://localhost:5050/#/experiments/2/runs/bc31007ca50c457dba1ea5b6fa1ea6d9
🧪 View experiment at: http://localhost:5050/#/experiments/2


##### ***Модель №5. Градиентный бустинг***


In [24]:
from catboost import CatBoostClassifier

params = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}

with mlflow.start_run(experiment_id=exp_id, run_name="Model #5. CatBoost Gradient Boosting"):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())

    model_cb = CatBoostClassifier(**params)
    model_cb.fit(X_train, y_train)

    y_pred = model_cb.predict(X_test)
    proba = model_cb.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall": recall_score(y_test, y_pred, pos_label=1),
        "f1_score": f1_score(y_test, y_pred, pos_label=1),
        "f2_score": fbeta_score(y_test, y_pred, pos_label=1, beta=2),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
    }

    conf_matrix = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("Предсказано")
    ax.set_ylabel("Истинно")
    ax.set_title("Матрица ошибок")
    plt.tight_layout()

    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    report = classification_report(y_test, y_pred)

    mlflow.log_text(report, "classification_report.txt")

    print("\nМатрица Ошибок")
    print(conf_matrix)

    print("\nОтчет по Классификации")
    print(report)

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test, model_cb.predict_proba(X_test))

    model_info = mlflow.catboost.log_model(
        cb_model=model_cb,
        artifact_path="model",
        signature=signature,
        registered_model_name=registered_model_name,
    )

    mlflow.log_input(
        mlflow.data.from_pandas(df, source="../data/raw/Fraud.csv"),
        context="training",
    )

    new_version = model_info.registered_model_version

    client.set_model_version_tag(
        registered_model_name,
        new_version,
        "env",
        "training",
    )

    client.set_registered_model_alias(
        registered_model_name,
        "training",
        new_version,
    )

    run_id = mlflow.active_run().info.run_id

    print("Run ID:", run_id)
    print("Registered model version:", new_version)
    print("Alias 'training' points to version:", new_version)
    print("Metrics:", metrics)

Artifact URI for this run: mlflow-artifacts:/fe15dbe84f8f4ad48f24487999bec679/artifacts

Матрица Ошибок
[[186540  75297]
 [  3925  48811]]

Отчет по Классификации
              precision    recall  f1-score   support

           0       0.98      0.71      0.82    261837
           1       0.39      0.93      0.55     52736

    accuracy                           0.75    314573
   macro avg       0.69      0.82      0.69    314573
weighted avg       0.88      0.75      0.78    314573



2026/05/25 22:17:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'fraud-detection-2' already exists. Creating a new version of this model...
2026/05/25 22:17:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-2, version 5
Created version '5' of model 'fraud-detection-2'.
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '../data/raw/Fraud.csv'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\Mi\Desktop\study\Search_for_anomalies_in_data\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: Loc

Run ID: fe15dbe84f8f4ad48f24487999bec679
Registered model version: 5
Alias 'training' points to version: 5
Metrics: {'accuracy': 0.7481602044676434, 'precision': 0.3932945499081445, 'recall': 0.9255726638349514, 'f1_score': 0.5520232521318224, 'f2_score': 0.7284093215381493, 'roc_auc': 0.8848964725605293, 'pr_auc': 0.5902905957845214}
🏃 View run Model #5. CatBoost Gradient Boosting at: http://localhost:5050/#/experiments/2/runs/fe15dbe84f8f4ad48f24487999bec679
🧪 View experiment at: http://localhost:5050/#/experiments/2


---

Подготовка модели для сохранения. Для выполнения задания выбрана модель №4 (RandomForestClassifier)

In [ ]:
import pickle
from pathlib import Path

# мы сейчас в notebooks/, поднимаемся в корень проекта
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "random_forest_model.pkl"

with open(MODEL_PATH, "wb") as f:
    pickle.dump(model_rf, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Модель сохранена: {MODEL_PATH}")
